In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [2]:
# Data Loading
# relative path from notebook/jaemuk to data/
file_path = "../../data/Telco_customer_churn - Telco_Churn.csv"
df = pd.read_csv(file_path)
print("Data loaded successfully.")
display(df.head())


Data loaded successfully.


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
# Data Cleaning: fillna for Churn Reason
df["Churn Reason"] = df["Churn Reason"].fillna("Not Disclosed")
print("Missing values handled.")


Missing values handled.


## Feature Engineering

In [4]:
# Feature Engineering

# 1. Target Encoding
df["Churn_Numeric"] = df["Churn Label"].apply(lambda x: 1 if x == "Yes" else 0)

# 2. Type Conversion and Derived Features
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0)
df["Is_Month_To_Month"] = df["Contract"].apply(lambda x: 1 if x == "Month-to-month" else 0)
df["No_Security_Support"] = ((df["Online Security"] == "No") & (df["Tech Support"] == "No")).astype(int)
df["New_Customer"] = df["Tenure Months"].apply(lambda x: 1 if x <= 6 else 0)
df["High_Risk_Fiber"] = ((df["Internet Service"] == "Fiber optic") & (df["Contract"] == "Month-to-month")).astype(int)


In [5]:
# Advanced Features
services = ["Online Security", "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies"]
df["Total_Services"] = df[services].apply(lambda x: (x == "Yes").sum(), axis=1)
df["Extra_Charges"] = df["Total Charges"] - (df["Monthly Charges"] * df["Tenure Months"])
df["Price_per_Service"] = df["Monthly Charges"] / (df["Total_Services"] + 1)
df["Full_Family"] = ((df["Partner"] == "Yes") & (df["Dependents"] == "Yes")).astype(int)
df["Single_Senior"] = ((df["Senior Citizen"] == "Yes") & (df["Partner"] == "No") & (df["Dependents"] == "No")).astype(int)
df["Auto_Payment"] = df["Payment Method"].str.contains("automatic").astype(int)


In [ ]:
df["Speed_Competitor_Risk"] = ((df["Internet Service"] == "Fiber optic") & (df["Contract"] == "Month-to-month") & (df["Tenure Months"].between(6, 24))).astype(int)
df["Price_Sensitive_Risk"] = ((df["Monthly Charges"] > df["Monthly Charges"].median()) & (df["Contract"] == "Month-to-month") & (df["Dependents"] == "No")).astype(int)
df["Tech_Lacking_Risk"] = ((df["Tech Support"] == "No") & (df["Online Security"] == "No") & (df["Internet Service"] != "No")).astype(int)
df["Early_Churn_Risk"] = ((df["Tenure Months"] <= 3) & (df["Contract"] == "Month-to-month")).astype(int)
df["Hidden_Fee_Flag"] = (df["Extra_Charges"] > 0).astype(int)
df["Loyal_But_Unbound"] = ((df["Tenure Months"] >= 24) & (df["Contract"] == "Month-to-month")).astype(int)


In [ ]:
df["Avg_Monthly_Bill"] = df["Total Charges"] / (df["Tenure Months"] + 1)
df["Streaming_Heavy_User"] = ((df["Streaming TV"] == "Yes") & (df["Streaming Movies"] == "Yes")).astype(int)
df["Engagement_Score"] = (
    (df["Contract"].isin(["One year", "Two year"])).astype(int) +
    (df["Payment Method"].str.contains("automatic")).astype(int) +
    ((df["Online Security"] == "Yes") | (df["Tech Support"] == "Yes")).astype(int)
)


In [9]:
output_path = "../../data/jm_preprocessed_churn_data.csv"
df.to_csv(output_path, index=False)
print(f"Preprocessed data saved to {output_path}")


Preprocessed data saved to ../../data/preprocessed_churn_data.csv
